# Project 1 — My experiment workbench

**You make the choices; this notebook makes them easy to test.** Follow the six questions in [the assignment](project.ipynb): data → tokenizer → features → architecture → evaluation → interpretation. Reuse [Tutorial 1](tutorial.ipynb) whenever a step is unfamiliar.

Start with one configuration on one fold. Write a prediction, run it, explain the result, then change **one setting**. Switch to five-fold CV when ready. This notebook does not automatically search for a winning model or fill in your conclusions.

**How to use it:** select the `.venv` Python kernel; run [setup.ipynb](setup.ipynb) if needed. Edit the control panel below, save the notebook, then **Restart Kernel and Run All**. This avoids stale variables after changing an earlier cell. Results are saved in a new local folder each run; the other notebooks and results are not overwritten. Nothing is published.

You can read slowly and run individual cells with Shift+Enter on your first pass. When changing settings for a recorded experiment, restart and run all.

## 1. My control panel

Edit the values in the next cell. These are ordinary Python variables, not a new framework.

| Setting | Examples | What changes? |
|---|---|---|
| `examples_per_class` | `500`, `2000`, `5000` | Training **and validation sample** size; different samples are not directly paired. |
| `vocab_size` | `1500`, `3000`, `8000` | BPE token inventory, not a document length limit. |
| `min_frequency` | `2`, `5` | Minimum pair frequency for a BPE merge. |
| `representation` | `"indicators"`, `"bow"`, `"tfidf"` | Which numbers represent the same BPE tokens. |
| `model_name` | `"logistic"`, `"mlp"` | Linear classifier or neural network. |
| `hidden_layers` | `(32,)`, `(64,)`, `(64, 32)` | MLP width and depth; the comma in `(64,)` matters. |
| `activation` | `"relu"`, `"tanh"`, `"gelu"` | Hidden-layer nonlinearity. |
| `dropout`, `weight_decay` | `0.2`, `0.0001` | MLP regularization. |
| `logistic_C` | `0.1`, `1.0`, `10.0` | Larger C means **weaker** logistic regularization. |
| `num_epochs`, `learning_rate` | `10`, `0.001` | Neural training budget and step size. |
| `run_all_folds` | `False` → `True` | One-fold learning run → five-fold evaluation. |
| `save_run` | `True` / `False` | Save a uniquely named local record. |

Neural settings do not affect logistic regression; C does not affect the MLP. Keep the data seed fixed when comparing models. The model seed can be changed in a later robustness experiment. The starting sample/vocabulary match your recent edits, not the original 1,500-token baseline.

In [ ]:
# DATA AND TOKENIZER
use_toy_data = False          # True: offline artificial topic demo, not harmfulness results
examples_per_class = 8000
seed = 42                    # controls sampling and folds
vocab_size = 8000
min_frequency = 3

# REPRESENTATION AND MODEL
representation = "tfidf"     # "indicators", "bow", "tfidf"
model_name = "mlp"           # "logistic", "mlp"
hidden_layers = (64,)        # try (32,) or (64, 32)
activation = "relu"          # "relu", "tanh", "gelu"
dropout = 0.2
weight_decay = 0.0001
logistic_C = 1.0

# TRAINING AND EVALUATION
num_epochs = 10
learning_rate = 0.001
batch_size = 64
model_seed = 42
run_all_folds = False        # False runs fold 0; True adds folds 1–4
save_run = True

# YOUR LAB NOTES: edit before running; do not put raw dataset prompts here.
experiment_name = "my_first_run"
my_hypothesis = ""           # Example: increasing vocabulary will reduce fragmentation.
what_i_changed = ""          # Example: only vocab_size, from 3000 to 8000.

### Before running: make a prediction

Double-click this Markdown cell and complete it in your own words:

- **What do I expect to improve, and why?** …
- **What might get worse?** …
- **Which result would contradict my prediction?** …
- **What am I holding fixed?** …

The text variables in the control panel are saved with the run. Markdown reflections stay in this notebook.

## 2. Setup and record the effective settings

The toy toggle deliberately uses a tiny sample and vocabulary. Its artificial topic labels are only for learning the pipeline. Real-data access errors are never silently replaced with toy data. The printed settings are what will actually run.

In [ ]:
from pathlib import Path
import os
import sys

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "part1/smlp.py").exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

# Keep generated caches in this local, Git-ignored directory.
os.environ.setdefault("MPLCONFIGDIR", str(root / ".cache/matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(root / ".cache"))
os.environ.setdefault("HF_HUB_CACHE", str(root / ".cache/huggingface/hub"))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

In [ ]:
import copy
import hashlib
import json
import random
import time
from datetime import datetime, timezone
from importlib.metadata import version
from uuid import uuid4

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

import part1
from utils.p1.functions import (
    TokenIndicators, BagOfWords, TFIDF, encode_texts,
    LearningMLP, train_mlp, predict_mlp, classification_scores,
)

torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
num_folds = 5
if use_toy_data:
    examples_per_class, vocab_size, num_epochs = 30, 128, 2
run_label = "TOY_NOT_HARMFULNESS_RESULTS" if use_toy_data else "EXPLORATORY_TRAIN_CV"
dataset_revision = "d29c47f41c8b51348b5c8e8c81c039b3132b66d1"
assert representation in ["indicators", "bow", "tfidf"]
assert model_name in ["logistic", "mlp"]
assert activation in ["relu", "tanh", "gelu"]
assert isinstance(hidden_layers, (tuple, list)) and len(hidden_layers) > 0
assert all(isinstance(width, int) and width > 0 for width in hidden_layers)
assert examples_per_class >= num_folds and vocab_size > 0 and min_frequency >= 1
assert 0 <= dropout < 1 and weight_decay >= 0 and logistic_C > 0
assert num_epochs > 0 and batch_size > 0 and learning_rate > 0

# Snapshot before fitting, so later control edits cannot silently relabel results.
setting_names = ["use_toy_data", "examples_per_class", "seed", "vocab_size", "min_frequency",
    "representation", "model_name", "hidden_layers", "activation", "dropout", "weight_decay",
    "logistic_C", "num_epochs", "learning_rate", "batch_size", "model_seed", "run_all_folds",
    "num_folds", "dataset_revision"]
settings = {name: copy.deepcopy(globals()[name]) for name in setting_names}
print(run_label)
display(pd.Series(settings, name="Effective settings"))

## 3. Understand and prepare the dataset

**Our task is fixed:** predict `prompt_harm_label` using only `prompt`, with 0 = unharmful and 1 = harmful. Subcategory only determines the agreed exclusion; responses and metadata are not features.

We load only the pinned `wildguardtrain/train` data using your existing login. The official test stays untouched. The following cells reuse the original notebook's audit and cleanup so a model experiment does not accidentally change the task.

We show counts and safe teaching examples. Raw dataset prompts are not exported or printed.

In [ ]:
if use_toy_data:
    records = []
    for i in range(examples_per_class):
        records.append({"prompt": f"Garden notebook {i}: water the flowers and prepare fresh soil.",
                        "prompt_harm_label": "unharmful", "subcategory": "synthetic"})
        records.append({"prompt": f"Space notebook {i}: observe the planets through a telescope tonight.",
                        "prompt_harm_label": "harmful", "subcategory": "synthetic"})
    data = pd.DataFrame(records)
else:
    dataset = load_dataset(
        "allenai/wildguardmix", "wildguardtrain", split="train",
        revision=dataset_revision, token=True,
        cache_dir=str(root / ".cache/huggingface/datasets"),
    )
    assert dataset.cache_files and all(
        dataset_revision in Path(item["filename"]).parts for item in dataset.cache_files
    ), "The cached dataset does not match our pinned revision."
    data = dataset.select_columns(["prompt", "prompt_harm_label", "subcategory"]).to_pandas()

print("Rows loaded:", len(data))

### Filter first, then audit labels

Exclude the agreed exact normalized category `toxic_language_hate_speech`. Check the expected raw-data counts before deduplication.

**Ask yourself:** why would removing a category change what conclusions we can claim?

In [ ]:
audit = {"status": run_label, "original_rows": len(data)}
excluded = data["subcategory"].fillna("").str.strip().str.casefold().eq("toxic_language_hate_speech")
audit["excluded_category_rows"] = int(excluded.sum())
data = data.loc[~excluded].copy()
assert not data["subcategory"].fillna("").str.strip().str.casefold().eq("toxic_language_hate_speech").any()

labels = data["prompt_harm_label"].str.strip().str.casefold()
missing_label = labels.isna() | labels.eq("")
audit["missing_label_rows"] = int(missing_label.sum())
assert labels.loc[~missing_label].isin(["unharmful", "harmful"]).all(), "Unexpected label."
data = data.loc[~missing_label].copy()
data["label"] = labels.loc[~missing_label].map({"unharmful": 0, "harmful": 1}).astype(int)
audit["eligible_rows"] = len(data)
audit["eligible_unharmful"] = int(data["label"].eq(0).sum())
audit["eligible_harmful"] = int(data["label"].eq(1).sum())

if not use_toy_data:
    expected = {"original_rows": 86759, "excluded_category_rows": 3020,
                "missing_label_rows": 0, "eligible_rows": 83739,
                "eligible_unharmful": 40543, "eligible_harmful": 43196}
    assert all(audit[key] == value for key, value in expected.items()), "Dataset audit mismatch."
display(pd.Series(audit, name="Data audit"))

### Remove empty prompts, contradictory labels and exact duplicates

Each removal has its own count. Exact deduplication prevents identical prompts crossing folds. Related scenarios and paraphrases can still cross folds.

**Ask yourself:** if the same text has two different labels, what information would a text-only classifier need to resolve that?

In [ ]:
data["prompt"] = data["prompt"].fillna("").str.normalize("NFC").str.strip()
empty_prompt = data["prompt"].eq("")
audit["empty_prompt_rows"] = int(empty_prompt.sum())
data = data.loc[~empty_prompt].copy()

conflicting = data.groupby("prompt")["label"].transform("nunique").gt(1)
audit["contradictory_label_rows"] = int(conflicting.sum())
audit["contradictory_label_groups"] = int(data.loc[conflicting, "prompt"].nunique())
data = data.loc[~conflicting].copy()
audit["duplicate_rows"] = int(data.duplicated("prompt").sum())
data = data.drop_duplicates("prompt").copy()
audit["retained_rows"] = len(data)

removals = ["excluded_category_rows", "missing_label_rows", "empty_prompt_rows",
            "contradictory_label_rows", "duplicate_rows", "retained_rows"]
assert sum(audit[key] for key in removals) == audit["original_rows"]
assert data["prompt"].is_unique

data["id"] = data["prompt"].map(lambda text: "wg-" + hashlib.sha256(text.encode("utf-8")).hexdigest())
data = data[["id", "prompt", "label"]].sort_values("id").reset_index(drop=True)
display(pd.Series(audit, name="Completed audit"))

### Sample and inspect distributions

We deliberately sample equally from both classes. This differs from natural prevalence. The labels used to stratify a split are not input features.

Changing sample size changes validation examples too. That is fine for exploration, but a proper training-size curve would keep its validation set fixed.

In [ ]:
rng = random.Random(seed)
chosen_indices = []
for label in [0, 1]:
    candidates = data.index[data["label"].eq(label)].tolist()
    assert len(candidates) >= examples_per_class
    chosen_indices.extend(rng.sample(candidates, examples_per_class))
rng.shuffle(chosen_indices)
sample = data.loc[chosen_indices].reset_index(drop=True)

texts = sample["prompt"].tolist()
y = sample["label"].to_numpy()
ids = sample["id"].to_numpy()
audit["sample_rows"] = len(sample)
audit["sample_harmful_prevalence"] = float(y.mean())
print("Sample class counts:", sample["label"].value_counts().sort_index().to_dict())
print("Eligible population harmful prevalence:", audit["eligible_harmful"] / audit["eligible_rows"])
display(sample["prompt"].str.len().describe(percentiles=[0.5, 0.95]).rename("Prompt length in characters"))
class_counts = pd.DataFrame({
    "cleaned_population": data.label.value_counts().sort_index(),
    "sample": sample.label.value_counts().sort_index(),
})
display(class_counts.rename(index={0: "unharmful", 1: "harmful"}))
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
class_counts.plot.bar(ax=axes[0], rot=0)
sample.prompt.str.len().plot.hist(bins=30, ax=axes[1])
axes[1].set_xlabel("Prompt characters")
axes[1].set_title("Sample lengths")
fig.tight_layout()
plt.show()

## 4. Split before learning features

The first fold is your quick development playground. Five folds are created immediately so switching on CV does not change their membership. Every fold learns its own BPE vocabulary and IDF from training text only.

**Ask yourself:** why is fitting a tokenizer on all 4,000 sampled prompts different from fitting it on a fold's 3,200 training prompts? (Numbers change with your settings.)

In [ ]:
splitter = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=seed)
folds = list(splitter.split(texts, y))
fold_assignment = np.full(len(sample), -1)
for fold, (train_indices, validation_indices) in enumerate(folds):
    assert set(train_indices).isdisjoint(validation_indices)
    fold_assignment[validation_indices] = fold
    print(f"Fold {fold + 1}: {len(train_indices)} train, {len(validation_indices)} validation")
assert (fold_assignment >= 0).all()

# Start with one fold to understand each component, like the tutorial.
train_indices, validation_indices = folds[0]
train_texts = [texts[i] for i in train_indices]
validation_texts = [texts[i] for i in validation_indices]
y_train, y_validation = y[train_indices], y[validation_indices]
split_table = pd.DataFrame({"id": ids, "gold": y, "fold": fold_assignment})
split_fingerprint = hashlib.sha256(split_table.to_csv(index=False).encode()).hexdigest()
print("Sample/fold fingerprint:", split_fingerprint[:12])

## 5. Tokenize: inspect the pieces yourself

BPE starts from characters and learns frequent merges. The Metaspace pretokenizer marks word boundaries with `▁`. Larger vocabularies permit more longer units; they do not guarantee better classification. An unseen word may use known pieces; an unseen character can become `[UNK]` in this implementation.

Change the **safe example** below to an ordinary sentence you choose. Do not treat one sentence's prediction or tokenization as an evaluation. Learn the tokenizer from the training fold, then inspect the sentence.

**Write down:** which words split, and how do the pieces change when you increase vocabulary size?

In [ ]:
tokenizer = part1.experiment.train_tokenizer(train_texts, vocab_size, min_frequency)
safe_example = "Please explain how plants grow in a garden."
print("Pieces:", tokenizer.encode(safe_example).tokens)
print("Requested vocabulary:", vocab_size)
print("Actual vocabulary:", tokenizer.get_vocab_size())
print("Example token count:", len(tokenizer.encode(safe_example).ids))

## 6. Turn tokens into features

These are **three representations of the same tokens**, not three tokenizer algorithms.

| Representation | For illustrative tokens `[garden, garden, plants]` | Question it answers |
|---|---|---|
| Indicators | garden: 1, plants: 1 | Did it appear? |
| Counts (`bow`) | garden: 2, plants: 1 | How often? |
| TF-IDF | count × IDF, followed by L2 normalization | How distinctive is it in this training corpus? |

Our smoothed IDF is `log((1 + n_train)/(1 + df_train)) + 1`. All three lose order. The actual BPE pieces below may split the example words.

**Predict before looking:** if you repeat a word three times, which columns change? Would repeating the entire sentence change a normalized TF-IDF vector?

In [ ]:
encoders = {
    "indicators": TokenIndicators(tokenizer),
    "bow": BagOfWords(tokenizer),
    "tfidf": TFIDF(tokenizer).fit(train_texts),
}
safe_repetition = "garden garden garden plants"
active = np.flatnonzero(encoders["bow"](safe_repetition))
display(pd.DataFrame({
    "token": [tokenizer.id_to_token(int(i)) for i in active],
    **{name: encoder(safe_repetition)[active] for name, encoder in encoders.items()},
}))
encoder = encoders[representation]
X_train = encode_texts(train_texts, encoder)
X_validation = encode_texts(validation_texts, encoder)
print("Chosen representation:", representation)
print("Train shape:", X_train.shape, "Validation shape:", X_validation.shape)
print("Fraction of nonzero training entries:", X_train.nnz / np.prod(X_train.shape))

## 7. Build one model

**Logistic:** features → linear score → class probabilities.

**MLP:** features → hidden layer(s) and activation → class probabilities. `(64, 32)` means two hidden layers, with 64 then 32 units. Dropout is active during training only. The course model returns log probabilities for stable NLL training.

`LearningMLP` in [functions.py](../utils/p1/functions.py) extends the tutorial's sparse MLP to expose widths and activation. Its sparse first layer and forward method are reused. ReLU clips negative inputs to zero; tanh maps values between −1 and 1; GELU smoothly weights inputs.

**Ask yourself:** how does increasing vocabulary size affect the first layer's parameter count? Can another hidden layer recover word order that our features discarded?

In [ ]:
fold = 0
if model_name == "logistic":
    model = LogisticRegression(C=logistic_C, solver="lbfgs", max_iter=1000,
                               random_state=model_seed + fold)
else:
    torch.manual_seed(model_seed + fold)
    model = LearningMLP(X_train.shape[1], hidden_layers, activation, dropout)
print(model)
if model_name == "mlp":
    print("Trainable parameters:", sum(p.numel() for p in model.parameters()))

## 8. Train and inspect

The helper `train_mlp` calls the course's dataset, batching and `train_one_epoch` functions. Its code is short enough to read alongside this cell. It uses Adam, NLL loss and a fixed epoch budget. We do not select an epoch using this validation fold.

For logistic regression the solver fits directly; there is no MLP learning-rate or dropout setting to apply.

**Ask yourself:** if training loss falls, must validation performance improve?

In [ ]:
started = time.perf_counter()
if model_name == "logistic":
    model.fit(X_train, y_train)
    history = []
else:
    history = train_mlp(model, X_train, y_train, epochs=num_epochs,
                        seed=model_seed + fold, batch_size=batch_size,
                        learning_rate=learning_rate, weight_decay=weight_decay)
fit_seconds = time.perf_counter() - started
print("Fit seconds (excluding feature construction):", round(fit_seconds, 3))
if history:
    training_history = pd.DataFrame(history)
    training_history.index = np.arange(1, len(history) + 1)
    training_history.index.name = "epoch"
    display(training_history)
    training_history.loss.plot(marker="o", ylabel="Training NLL", xlabel="Epoch")
    plt.show()
    print("History accuracy is measured during training with dropout; it is not held-out accuracy.")

## 9. Evaluate on unseen rows from this fold

Macro-F1 averages the two class F1 scores. Accuracy counts correct predictions. Harmful recall measures how many harmful examples we catch; benign false-positive rate measures how often we flag unharmful examples.

The model chooses its most probable class. We also evaluate the final model on its training rows with dropout disabled, so the train–validation comparison uses the same prediction mode.

**Write down:** which kind of error matters more for your intended use, and what trade-off would you accept? A high score here does not measure deployed-agent safety.

In [ ]:
if model_name == "logistic":
    probabilities = model.predict_proba(X_validation)
    train_probabilities = model.predict_proba(X_train)
else:
    probabilities = predict_mlp(model, X_validation)
    train_probabilities = predict_mlp(model, X_train)
assert np.isfinite(probabilities).all()
np.testing.assert_allclose(probabilities.sum(axis=1), 1, atol=1e-6)
predictions = probabilities.argmax(axis=1)
metrics = classification_scores(y_validation, predictions)
training_metrics = classification_scores(y_train, train_probabilities.argmax(axis=1))
display(pd.DataFrame({"training": training_metrics, "validation": metrics}).T)
display(pd.DataFrame([[metrics["tn"], metrics["fp"]], [metrics["fn"], metrics["tp"]]],
    index=["Gold unharmful", "Gold harmful"],
    columns=["Predicted unharmful", "Predicted harmful"]))
print("Macro-F1 training minus validation:", training_metrics["macro_f1"] - metrics["macro_f1"])

# Keep this first-fold result before the optional CV loop replaces temporary variables.
score_rows = [{"fold": 0, **metrics, "training_macro_f1": training_metrics["macro_f1"],
               "fit_seconds": fit_seconds, "actual_vocab": tokenizer.get_vocab_size()}]
prediction_tables = [pd.DataFrame({"id": ids[validation_indices], "fold": 0,
    "gold": y_validation, "prediction": predictions, "probability_harmful": probabilities[:, 1]})]
first_fold_history = pd.DataFrame(history)
trained_settings = copy.deepcopy(settings)

### My explanation after the first fold

- **My prediction was supported / contradicted / still unclear because:** …
- **The training–validation gap suggests:** …
- **I should not conclude … because …**
- **The next single setting I want to change is:** …

A single fold is useful for learning. Repeatedly tuning against it makes it development data, not a fresh test. Do not claim a final winner from this cell.

## 10. Optional: evaluate this configuration on all five folds

Set `run_all_folds = True` in the control panel and restart/run all. We reuse fold 0's result and train fresh models and feature encoders for folds 1–4. This evaluates **one chosen configuration**, not a large automatic search.

One-fold and five-fold runs are kept separate when browsing history. CV folds overlap in their training data, so their standard deviation is descriptive rather than an independent-sample confidence interval.

In [ ]:
if run_all_folds:
    for fold, (train_indices, validation_indices) in enumerate(folds[1:], start=1):
        train_texts = [texts[i] for i in train_indices]
        validation_texts = [texts[i] for i in validation_indices]
        y_train, y_validation = y[train_indices], y[validation_indices]
        tokenizer = part1.experiment.train_tokenizer(train_texts, vocab_size, min_frequency)
        if representation == "indicators":
            encoder = TokenIndicators(tokenizer)
        elif representation == "bow":
            encoder = BagOfWords(tokenizer)
        else:
            encoder = TFIDF(tokenizer).fit(train_texts)
        X_train = encode_texts(train_texts, encoder)
        X_validation = encode_texts(validation_texts, encoder)

        if model_name == "logistic":
            model = LogisticRegression(C=logistic_C, solver="lbfgs", max_iter=1000,
                                       random_state=model_seed + fold)
        else:
            torch.manual_seed(model_seed + fold)
            model = LearningMLP(X_train.shape[1], hidden_layers, activation, dropout)
        started = time.perf_counter()
        if model_name == "logistic":
            model.fit(X_train, y_train)
            history = []
        else:
            history = train_mlp(model, X_train, y_train, epochs=num_epochs,
                                seed=model_seed + fold, batch_size=batch_size,
                                learning_rate=learning_rate, weight_decay=weight_decay)
        fit_seconds = time.perf_counter() - started
        if model_name == "logistic":
            probabilities = model.predict_proba(X_validation)
            train_probabilities = model.predict_proba(X_train)
        else:
            probabilities = predict_mlp(model, X_validation)
            train_probabilities = predict_mlp(model, X_train)
        assert np.isfinite(probabilities).all()
        np.testing.assert_allclose(probabilities.sum(axis=1), 1, atol=1e-6)
        predictions = probabilities.argmax(axis=1)
        metrics = classification_scores(y_validation, predictions)
        training_metrics = classification_scores(y_train, train_probabilities.argmax(axis=1))
        score_rows.append({"fold": fold, **metrics,
            "training_macro_f1": training_metrics["macro_f1"],
            "fit_seconds": fit_seconds, "actual_vocab": tokenizer.get_vocab_size()})
        prediction_tables.append(pd.DataFrame({"id": ids[validation_indices], "fold": fold,
            "gold": y_validation, "prediction": predictions,
            "probability_harmful": probabilities[:, 1]}))
        print("Completed fold", fold + 1)
else:
    print("One-fold learning mode. Enable run_all_folds for five-fold evaluation.")

scores = pd.DataFrame(score_rows)
oof = pd.concat(prediction_tables, ignore_index=True)
expected_folds = 5 if run_all_folds else 1
assert scores.fold.tolist() == list(range(expected_folds))
assert not oof.id.duplicated().any()
assert len(oof) == (len(sample) if run_all_folds else len(folds[0][1]))
metric_names = ["macro_f1", "accuracy", "harmful_recall", "benign_fpr", "training_macro_f1"]
display(scores[["fold", *metric_names]].round(4))
display(scores[metric_names].agg(["mean", "std"]).round(4))
print("One-fold std is undefined; five-fold std is not a confidence interval.")

## 11. Record an experiment without overwriting earlier runs

Update the reflection variables below after inspecting the results, then run the save cell. If you simply Run All, blank reflections are allowed; finish the Markdown notes afterwards. Notes should contain your reasoning, not raw dataset prompts.

Each saved folder contains settings, notes, data audit, fold scores, hashed-ID predictions, split membership, first-fold training history, package versions and source hashes. Save this notebook before running so its source on disk reflects your edits. Sources are copied into the run folder for reproducibility. Saved results are local and Git-ignored.

Changing any control after training requires a new run. The save cell checks this to avoid attaching new settings to old scores.

In [ ]:
my_observation = ""
my_interpretation = ""
my_next_experiment = ""

In [ ]:
current_settings = {name: copy.deepcopy(globals()[name]) for name in setting_names}
assert current_settings == trained_settings, "Settings changed: restart and run all before saving."
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + "_" + uuid4().hex[:6]
run_root = root / "part1/results/learning" / ("toy" if use_toy_data else "real")
notes = {"experiment_name": experiment_name, "hypothesis": my_hypothesis,
    "change": what_i_changed, "observation": my_observation,
    "interpretation": my_interpretation, "next_experiment": my_next_experiment,
    "AI_assistance": "Codex created the initial workbench, configurable MLP helper and verification. "
                     "I choose settings, formulate hypotheses, run experiments and review conclusions."}
record = {"run_id": run_id, "status": run_label, "settings": trained_settings,
    "split_fingerprint": split_fingerprint, "evaluated_folds": scores.fold.tolist(),
    "official_test_accessed": False, "notes": notes,
    "packages": {name: version(name) for name in
                 ["numpy", "pandas", "scipy", "scikit-learn", "torch", "tokenizers", "datasets", "baycomp", "nbformat"]}}
if save_run:
    run_folder = run_root / run_id
    run_folder.mkdir(parents=True, exist_ok=False)
    scores.to_csv(run_folder / "scores.csv", index=False)
    oof.to_csv(run_folder / "predictions.csv", index=False)
    split_table.to_csv(run_folder / "splits.csv", index=False)
    first_fold_history.to_csv(run_folder / "first_fold_history.csv", index=False)
    (run_folder / "data_audit.json").write_text(json.dumps(audit, indent=2))
    # Save an output-free notebook and the small code dependencies used by this run.
    import nbformat
    source_folder = run_folder / "source"
    dependencies = ["part1/project1_learning.ipynb", "utils/p1/functions.py",
                    "utils/__init__.py", "utils/p1/__init__.py"]
    dependencies += [str(p.relative_to(root)) for p in (root / "part1").glob("*.py")]
    record["source_sha256"] = {}
    for relative in dependencies:
        source_path = root / relative
        target = source_folder / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        if source_path.suffix == ".ipynb":
            snapshot = nbformat.read(source_path, 4)
            for cell in snapshot.cells:
                if cell.cell_type == "code":
                    cell.outputs, cell.execution_count = [], None
            nbformat.write(snapshot, target)
        else:
            target.write_bytes(source_path.read_bytes())
        record["source_sha256"][relative] = hashlib.sha256(target.read_bytes()).hexdigest()
    (run_folder / "record.json").write_text(json.dumps(record, indent=2))
    print("Saved new run:", run_folder)
else:
    print("Saving disabled. No run record was written.")

## 12. Compare my saved runs fairly

The history table below only includes runs with exactly the same sample/fold fingerprint, dataset revision and evaluated folds. It therefore does not silently compare a one-fold score against a five-fold mean or a different sample. Other runs remain on disk.

A ranking is descriptive. Repeated experimentation on these same folds means they are development evidence; a chosen model needs a frozen configuration and fresh evaluation later. Record failed hypotheses too.

In [ ]:
history_rows = []
for record_path in sorted(run_root.glob("*/record.json")):
    previous = json.loads(record_path.read_text())
    if (previous["split_fingerprint"] != split_fingerprint
        or previous["evaluated_folds"] != scores.fold.tolist()
        or previous["settings"]["dataset_revision"] != dataset_revision):
        continue
    previous_scores = pd.read_csv(record_path.parent / "scores.csv")
    s = previous["settings"]
    history_rows.append({"run_id": previous["run_id"],
        "name": previous["notes"]["experiment_name"],
        "vocab": s["vocab_size"], "min_frequency": s["min_frequency"], "representation": s["representation"],
        "model": s["model_name"], "layers": str(s["hidden_layers"]),
        "activation": s["activation"], "dropout": s["dropout"],
        "weight_decay": s["weight_decay"], "C": s["logistic_C"],
        "epochs": s["num_epochs"], "learning_rate": s["learning_rate"],
        "batch_size": s["batch_size"], "model_seed": s["model_seed"],
        "macro_f1": previous_scores.macro_f1.mean(),
        "accuracy": previous_scores.accuracy.mean(),
        "harmful_recall": previous_scores.harmful_recall.mean(),
        "benign_fpr": previous_scores.benign_fpr.mean()})
run_history = pd.DataFrame(history_rows)
if len(run_history):
    display(run_history.sort_values("macro_f1", ascending=False).round(4))
else:
    print("No saved runs with matching evaluation rows yet.")

## 13. Optional: Bayesian comparison of two five-fold runs

After saving two compatible five-fold runs, paste their `run_id` values below. Choose the comparison and ROPE **before** looking at its posterior when possible. Our default ROPE is ±0.01 macro-F1: a one-point change is treated as practically small for this project, not as a universal safety threshold.

The tutorial's model uses **B − A**. Positive differences favour B. We check identical split IDs, labels and fold order, not just similar scores. The model accounts for correlated CV under its assumptions; five folds are still limited evidence, and it does not remove bias from searching many configurations.

Leave both IDs blank to skip. This step needs five-fold runs, not the one-fold learning mode.

In [ ]:
comparison_run_a = ""
comparison_run_b = ""
rope = 0.01

if comparison_run_a and comparison_run_b:
    # IDs must be folder names under this workbench's real/toy result directory.
    assert all(Path(x).name == x and x not in [".", ".."] for x in [comparison_run_a, comparison_run_b])
    assert comparison_run_a != comparison_run_b and rope > 0
    folder_a, folder_b = run_root / comparison_run_a, run_root / comparison_run_b
    a_record = json.loads((folder_a / "record.json").read_text())
    b_record = json.loads((folder_b / "record.json").read_text())
    assert a_record["evaluated_folds"] == b_record["evaluated_folds"] == list(range(5)), "Use two five-fold runs."
    assert a_record["settings"]["dataset_revision"] == b_record["settings"]["dataset_revision"]
    pd.testing.assert_frame_equal(pd.read_csv(folder_a / "splits.csv"), pd.read_csv(folder_b / "splits.csv"))
    a = pd.read_csv(folder_a / "scores.csv").sort_values("fold")
    b = pd.read_csv(folder_b / "scores.csv").sort_values("fold")
    assert a.fold.tolist() == b.fold.tolist() == list(range(5))
    differences = b.macro_f1.to_numpy() - a.macro_f1.to_numpy()
    if np.allclose(differences, differences[0]):
        delta = float(differences.mean())
        equivalent = abs(delta) <= rope or np.isclose(abs(delta), rope)
        probabilities = [float(delta < -rope and not equivalent), float(equivalent),
                         float(delta > rope and not equivalent)]
        print("Constant fold differences: point-mass convention, not a fitted t distribution.")
    else:
        posterior = part1.analysis.BenavoliCorrelatedT(a.macro_f1.to_numpy(), b.macro_f1.to_numpy(),
                                                      runs=1, names=["A", "B"])
        probabilities = posterior.probs(rope)
        fig, ax = posterior.plot(rope)
        plt.show()
    comparison = {"A": comparison_run_a, "B": comparison_run_b, "rope": rope,
                  "mean_B_minus_A": float(differences.mean()),
                  **dict(zip(["P_A_wins", "P_equivalent", "P_B_wins"], map(float, probabilities)))}
    display(pd.Series(comparison))
    if save_run:
        comparison_path = run_root / ("comparison_" + uuid4().hex + ".json")
        comparison_path.write_text(json.dumps(comparison, indent=2))
        print("Saved comparison:", comparison_path.name)
else:
    print("Optional comparison skipped. Supply two saved five-fold run IDs when ready.")

## 14. My next experiments and submission checklist

Try these **one at a time**, writing a prediction before each:

1. Keep logistic regression fixed; compare indicators, counts and TF-IDF.
2. Keep representation/model fixed; compare 1,500, 3,000 and 8,000 BPE tokens.
3. Keep everything else fixed; change one hidden layer `(64,)` to two `(64, 32)`.
4. Keep layers fixed; try another activation, then a separate dropout experiment.
5. Keep the chosen configuration fixed; repeat with another model seed to assess initialization sensitivity.

These are suggestions, not prewritten conclusions. You can choose a different hypothesis.

For the required project comparison, use five folds for all nine combinations of **three representations × logistic / one-layer MLP / modified MLP**, keeping sample, tokenizer settings and fold membership fixed. Then use the paired Bayesian comparison for clearly motivated contrasts. The original [project1.ipynb](project1.ipynb) contains a complete 3 × 3 example if you need to check your understanding.

- [ ] I can explain the labels, filtering, class counts and sampling.
- [ ] I know where the tokenizer and IDF are fitted.
- [ ] I can explain indicators, counts and TF-IDF without reading the code.
- [ ] I can draw the architecture I tested and explain the setting I changed.
- [ ] I completed the required CV comparisons, including unsuccessful experiments.
- [ ] I can explain the metrics, B − A sign, ROPE and uncertainty.
- [ ] My conclusions separate measured results from possible explanations.
- [ ] My notes record sources and AI assistance, and I can reproduce a saved run.

**My final conclusion, in my own words:** …

**Attribution and AI record:** course BPE, sparse MLP, training utilities and Bayesian analysis are reused. Existing project helpers supply counts, TF-IDF and metrics. Codex created the initial notebook scaffold, configurable MLP helper and verification. Record any further assistance here, including what you accepted, changed or checked yourself. You retain responsibility for the experiments and explanation.